# <u>Nested Resampling</u>

### Prerequisites:

* <a href="../1.Supervised Learning/Evaluation.ipynb">Check out the notebook on Evaluation</a>
* <a href="../1.Supervised Learning/Hyperparameter Tuning.ipynb">Check out the notebook on Hyperparameter Tuning</a>

    
## Topics

* [1. Why Nested Resampling is Needed](#why)
* [2. The Untouched Test Set Principle](#test)
* [3. Simplest Version: Train / Validation / Test Split](#split)
* [4. Why this still isn't Ideal](#ideal)
* [5. What Nested Resampling Is](#nested)
* [6. Visual Interpretation of the Procedure](#visual)
* [7. Main Benefit](#benefit)

In [2]:
import numpy as np # for random numbers and arrays

import matplotlib.pyplot as plt # for plotting
import plotly.express as px # for plotting

# Create Datasets
from sklearn.datasets import make_regression # create toy data for Regression
from sklearn.datasets import make_classification # create toy data for Classification

from sklearn.model_selection import train_test_split # split dataset into train and test set


# Metrics for Regression
from sklearn.metrics import (
    mean_squared_error, # MSE
    mean_absolute_error, # MAE
    mean_absolute_percentage_error, # MAPE
    r2_score # R^2
)

# Metric for Classification
from sklearn.metrics import (
    accuracy_score, # Accuracy
    brier_score_loss, # Brier Score
    log_loss, # Log Loss
    confusion_matrix, # TP, FP, FN, TN
    precision_score, # Precision = PPV
    recall_score, # Recall = TPR
    roc_auc_score, # Area under ROC curve
    roc_curve, # ROC curve
)

from sklearn.preprocessing import StandardScaler # standardize features

# Nested resampling
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score # runs cross-validation and returns just the scores per fold
from sklearn.model_selection import cross_validate # more advanced version of cross_val_score that returns multiple metrics and extra info
from sklearn.model_selection import cross_val_predict # Generates predictions for each data point using cross-validation

from sklearn.pipeline import Pipeline # pipeline


# ML algorithms
from sklearn.neighbors import KNeighborsClassifier # k neares neighbors
from sklearn.tree import DecisionTreeClassifier # for Classification trees
from sklearn.tree import DecisionTreeRegressor # for Regression trees
from sklearn.svm import SVC # (non) linear (non)separable SVM
from sklearn.linear_model import LogisticRegression # logistic regression
from sklearn.ensemble import RandomForestClassifier # Random forest classifier

print("Setup complete")

Setup complete


<a class="anchor" id="why"></a>
# 1. Why Nested Resampling is Needed

In machine learning, we usually do **model selection**:

- choosing among different algorithms,
- tuning hyperparameters,
- selecting features,
- choosing preprocessing steps.

A major problem occurs when we use the **same resampling/test splits** both to tune the model and **to estimate its final performance**.

What goes wrong?

Each time we try many hyperparameter settings, we pick the one with the lowest cross-validation error.

But that "best" CV score is partly due to chance.

So:

>The model becomes fitted not only to the training data, but also to the validation/test splits used during tuning.

This is called:

- overtuning
- information leakage
- optimistic bias in performance estimation

The reported accuracy/error then looks **better than the true generalization performance**.


<a class="anchor" id="test"></a>
# 2. The Untouched Test Set Principle

To obtain an honest estimate of performance:
>The final test set must remain completely untouched during model building.

That means the test data cannot influence:

- hyperparameter tuning,
- feature engineering,
- preprocessing choices,
- model family selection.

The test set is used only once:
>After every modeling decision has already been made.

Why?: Because only then does test performance simulate:

>"How well would this model perform on truly unseen future data?"

This is the central principle behind nested resampling.

<a class="anchor" id="split"></a>
# 3. Simplest Version: Train / Validation / Test Split

### Training set
- Used to fit models.

### Validation set
- Used for tuning hyperparameters $\lambda$. Train many models with different $\lambda$ and choose the best hyperparameter setting denoted as **$\lambda^*$**

### (Untouched) Test set
- Retrain the model on training + validation data
- Evaluate **$\lambda^*$** once on the untouched test set

**This gives an approximately unbiased performance estimate.**

<p align="center">
<img src="pics/38.png" width="600"/>
</p>


In [45]:
X, y = make_classification(n_samples=300,n_features=3,n_classes=4,n_informative=2,n_redundant=1,n_clusters_per_class=1,random_state=1421)
labels = np.array([f"Class {i+1}" for i in y])
fig = px.scatter_3d(x=X[:,0],y=X[:,1],z=X[:,2],color=labels,labels={"color":"Classes","x":"x1","y":"x2","z":"x3"},title="Data")
fig.show()


# Untouched Test Split
X_train_full, X_untouched_test, y_train_full, y_untouched_test = train_test_split(X, y, test_size=0.25, random_state=1421)


# Hyperparameter candidates
k_vals = np.arange(1,11)
ensemble_sizes = np.arange(10,21)

ml_models_loss = {"kNN": [],"Random Forest Classifier": []}

kfold = KFold(n_splits=3, shuffle=True, random_state=1421)


# Tune KNN
for k in k_vals:
    
    model = KNeighborsClassifier(n_neighbors=k)
    fold_scores = []
    
    for train_idx, val_idx in kfold.split(X_train_full):
        X_train = X_train_full[train_idx]
        X_valid = X_train_full[val_idx]
        y_train = y_train_full[train_idx]
        y_valid = y_train_full[val_idx]

        model.fit(X_train, y_train)
        y_preds = model.predict(X_valid)

        fold_scores.append(accuracy_score(y_valid, y_preds))
    
    ml_models_loss["kNN"].append(np.mean(fold_scores))


# Tune Random Forest
for ensemble in ensemble_sizes:
    
    model = RandomForestClassifier(n_estimators=ensemble, random_state=1421)
    fold_scores = []
    
    for train_idx, val_idx in kfold.split(X_train_full):
        X_train = X_train_full[train_idx]
        X_valid = X_train_full[val_idx]
        y_train = y_train_full[train_idx]
        y_valid = y_train_full[val_idx]

        model.fit(X_train, y_train)
        y_preds = model.predict(X_valid)

        fold_scores.append(accuracy_score(y_valid, y_preds))
    
    ml_models_loss["Random Forest Classifier"].append(np.mean(fold_scores))

# choose best hyperparameters λ*
best_k = k_vals[np.argmax(ml_models_loss["kNN"])]
best_ensemble = ensemble_sizes[np.argmax(ml_models_loss["Random Forest Classifier"])]

print("Best k:", best_k)
print("Best ensemble size:", best_ensemble)

best_knn_score = np.max(ml_models_loss["kNN"])
best_rf_score = np.max(ml_models_loss["Random Forest Classifier"])

print(f"best_knn_score={best_knn_score},best_rf_score={best_rf_score}")

# Retrain λ* on train_full and evaluate once on untouched test
final_models = {"kNN": KNeighborsClassifier(n_neighbors=best_k),
                "Random Forest Classifier": RandomForestClassifier(n_estimators=best_ensemble, random_state=1421)}

for name, model in final_models.items():
    model.fit(X_train_full, y_train_full)
    y_preds = model.predict(X_untouched_test)

    test_acc = accuracy_score(y_untouched_test, y_preds)
    print(name, "Untouched Test Accuracy:", test_acc)

Best k: 9
Best ensemble size: 17
best_knn_score=0.888888888888889,best_rf_score=0.88
kNN Untouched Test Accuracy: 0.88
Random Forest Classifier Untouched Test Accuracy: 0.92


In [ ]:
X, y = make_classification(n_samples=300,n_features=3,n_classes=4,n_informative=2,n_redundant=1,n_clusters_per_class=1,random_state=1421)
labels = np.array([f"Class {i+1}" for i in y])
fig = px.scatter_3d(x=X[:,0],y=X[:,1],z=X[:,2],color=labels,labels={"color":"Classes","x":"x1","y":"x2","z":"x3"},title="Data")
fig.show()


# Untouched test split
X_train_full, X_untouched_test, y_train_full, y_untouched_test = train_test_split(X, y, test_size=0.25, random_state=1421)


# Hyperparameter candidates
k_vals = np.arange(1,11)
ensemble_sizes = np.arange(10,21)

ml_models_loss = {"kNN": [],"Random Forest Classifier": []}

kfold = KFold(n_splits=3, shuffle=True, random_state=1421)

# Tuning Loop 
for k, ensemble in zip(k_vals, ensemble_sizes): # Note: This does not evaluate Hyperparameters independently

    ml_models = {"kNN": KNeighborsClassifier(n_neighbors=k),
                 "Random Forest Classifier": RandomForestClassifier(n_estimators=ensemble,random_state=1421)}

    for name, model in ml_models.items():

        loss_metric_per_fold = []

        for train_idx, val_idx in kfold.split(X_train_full):

            X_train = X_train_full[train_idx]
            X_valid = X_train_full[val_idx]
            y_train = y_train_full[train_idx]
            y_valid = y_train_full[val_idx]

            model.fit(X_train, y_train)
            y_preds = model.predict(X_valid)

            acc = accuracy_score(y_valid, y_preds)
            loss_metric_per_fold.append(acc)

        ml_models_loss[name].append(np.mean(loss_metric_per_fold))


# Select λ* for each learner
best_k = k_vals[np.argmax(ml_models_loss["kNN"])]
best_ensemble = ensemble_sizes[np.argmax(ml_models_loss["Random Forest Classifier"])]
print("Best k:", best_k)
print("Best ensemble:", best_ensemble)

# Retrain the tuned models on full train_full
best_models = {"kNN": KNeighborsClassifier(n_neighbors=best_k),
               "Random Forest Classifier": RandomForestClassifier(n_estimators=best_ensemble,random_state=1421)}

for name, model in best_models.items():
    model.fit(X_train_full, y_train_full)
    y_preds = model.predict(X_untouched_test)

    test_acc = accuracy_score(y_untouched_test, y_preds)
    print(name, "Untouched Test Accuracy:", test_acc)

Best k: 9
Best ensemble: 17
kNN Untouched Test Accuracy: 0.88
Random Forest Classifier Untouched Test Accuracy: 0.92


<a class="anchor" id="ideal"></a>
# 4. Why this still isn't ideal

**A single 3-way split has drawbacks:**
- wastes data,
- estimate depends heavily on one lucky/unlucky split,
- unstable for small datasets.

So just as ordinary holdout can be improved by cross-validation, the 3-way split is generalized to: <u>Nested Resampling</u>


<a class="anchor" id="nested"></a>
# 5. What Nested Resampling is

Nested resampling uses two loops of resampling:

#### Outer Loop = Honest Performance Estimation

This loop creates outer train/test splits.

For each outer iteration:

- one part is held out as an untouched outer test fold,
- the remaining data goes to model building.

This outer test fold is never used during tuning.

Its only purpose:
> estimate final generalization error.

---

#### Inner Loop = Hyperparameter Tuning

Inside each outer training split, we now perform another resampling procedure:

usually k-fold CV.

For each candidate hyperparameter λᵢ:

- evaluate via inner CV,
- compute average validation performance.

Choose:

>$\lambda^*$ = best hyperparameter setting.

---

#### Then:

After $\lambda^*$ is selected:

- retrain on the full outer training set,
- evaluate once on the outer test fold.

Store that outer test performance.Repeat this for every outer fold.

---

#### Final Result

Average all outer test performances.That average is the nested resampling estimate.

Because every outer test fold was untouched by tuning:

>this estimate is approximately unbiased.


<a class="anchor" id="visual"></a>
# 6. Visual Interpretation of the Procedure

For each outer split:

1. Hold out outer test data.
2. On remaining data:
    - run inner CV,
    - compare all hyperparameters.
3. Select best $\lambda^*$.
4. Retrain using $\lambda^*$ on all outer training data.
5. Predict on untouched outer test fold.
6. Save outer error.

Then average all saved outer errors.

So nested resampling treats:

>"hyperparameter tuning as part of the training algorithm itself."

We are no longer evaluating a manually tuned learner. buth rather 
>a full self-tuning modeling procedure.

<div style="display: flex; justify-content: center; gap: 10px;">
  <img src="pics/39.png" width="550"/>
  <img src="pics/47.jpg" width="550"/>
</div>

In [55]:
X, y = make_classification(n_samples=500,n_features=2,n_classes=3,n_informative=2,n_redundant=0,n_clusters_per_class=1,class_sep=0.8,flip_y=0.08,random_state=1517)
labels = np.array([f"Class {i+1}" for i in y])
fig = px.scatter(x=X[:,0],y=X[:,1],color=labels,labels={"color":"Classes","x":"x1","y":"x2"},title="Data")
fig.show()


# Nested Resampling Setup
#outer_kfold = KFold(n_splits=3, shuffle=True, random_state=1517)
#inner_kfold = KFold(n_splits=4, shuffle=True, random_state=1517)
outer_kfold = StratifiedKFold(n_splits=3, shuffle=True, random_state=1517)
inner_kfold = StratifiedKFold(n_splits=4, shuffle=True, random_state=1517)

num_trees = np.arange(10,21)

outer_rf_error = []
best_hps_outer = []

# Outer loop
for outer_train_idx, outer_test_idx in outer_kfold.split(X,y): #outer_kfold.split(X):

    X_outer_train = X[outer_train_idx]
    X_outer_test = X[outer_test_idx]
    y_outer_train = y[outer_train_idx]
    y_outer_test = y[outer_test_idx]

    error_avg_per_hp = []

    # Inner Loop = hyperparameter tuning
    for n_tree in num_trees:

        rf = RandomForestClassifier(n_estimators=n_tree,random_state=1517)

        inner_fold_errors = []

        for inner_train_idx, inner_test_idx in inner_kfold.split(X_outer_train,y_outer_train): #inner_kfold.split(X_outer_train):

            X_inner_train = X_outer_train[inner_train_idx]
            X_inner_test = X_outer_train[inner_test_idx]
            y_inner_train = y_outer_train[inner_train_idx]
            y_inner_test = y_outer_train[inner_test_idx]

            rf.fit(X_inner_train, y_inner_train)
            y_preds = rf.predict(X_inner_test)

            fold_error = 1 - accuracy_score(y_inner_test, y_preds)
            inner_fold_errors.append(fold_error)

        error_avg_per_hp.append(np.mean(inner_fold_errors))


    # choose lambda*
    best_hp = num_trees[np.argmin(error_avg_per_hp)]
    best_hps_outer.append(best_hp)


    # retrain on full outer train using lambda*
    best_rf = RandomForestClassifier(n_estimators=best_hp,random_state=1517)

    best_rf.fit(X_outer_train, y_outer_train)
    y_outer_preds = best_rf.predict(X_outer_test)

    outer_error = 1 - accuracy_score(y_outer_test, y_outer_preds)
    outer_rf_error.append(outer_error)

print("Best hyperparameter chosen in each outer fold:", best_hps_outer)
print("Outer fold errors:", outer_rf_error)
print("Nested Resampling Estimated Generalization Error:", np.mean(outer_rf_error))

Best hyperparameter chosen in each outer fold: [np.int64(11), np.int64(13), np.int64(16)]
Outer fold errors: [0.23353293413173648, 0.22754491017964074, 0.2168674698795181]
Nested Resampling Estimated Generalization Error: 0.2259817713969651


<a class="anchor" id="benefit"></a>
# 7. Main Benefit

Without nested resampling:

- tuning score is too optimistic.

With nested resampling:

- outer scores reflect true unseen-data performance.


### Final Takeaway (One-Sentence)

>Nested resampling is a two-level validation strategy that separates hyperparameter tuning from final performance estimation, preventing overtuning and giving an unbiased estimate of generalization error.